# Homework-4
Recurrent Neural Networks  

## Problem 1:

Answer each of the following questions.

(A) Both CNNs and RNNs share parameters-but in different ways. Explain.

> Both CNNs and RNNs apply filters/kernels to the data, but across different dimensions. A CNN will apply the filter across space, scanning through every pixel position in an image. Meanwhile, RNNs will apply the filter across time, scanning across every time step in the sample. 


(B) What is the basic idea underlying how we learn meaningful word embeddings? How does this relate to the idea of self-supervised learning?

> We learn meaningful word embeddings from the context in which they are found. When we train a model, we have it predict words based on the context which allow the internal representations that the model develops to solve that task and encode semantic structure. As such, the data becomes self-supervised since the training signal comes from the data itself rather than a person labelling it by hand.

(C) Consider an LSTM layer that accepts a batch of arbitrary size, with 40 time steps and 3 features. The LSTM layer has input shape `[None, 40, 3]`. Suppose its output is shape `[None, 2]`. Find the number of parameters in the LSTM layer, explaining your thinking.

> In an LSTM, there are four gates (input, forget, output, and cell). Within each gate, there is are input weights that is 2x3, hidden weights that are 2x2, and a bias term that is 2x1. Adding these all up there are 12 parameters in each gate, times 4 we get 48 parameters in one LSTM layer. 

## Problem-2: 

 Begin by reviewing Andrej Karpathy's famous blog post [The Unreasonable Effectiveness of Recurrent Neural Networks](http://karpathy.github.io/2015/05/21/rnn-effectiveness/).

Provide a three-paragraph summary of the article, describing Karpathy's main points.

The article "The Unreasonable Effectiveness of Recurrent Neural Networks" by Andrej Karpathy summarizes what a Recurrent Neural Network (RNN) is and why they are such a powerful tool. Unlike a typical neural network that uses a fixed-size input and output, RNNs opwerate over sequences, allowing them to handle different length inputs and/or outputs. Furthermore, they contain a hidden state which transfers information from previous inputs into later ones, meaning the network's response isn't just shaped by what it sees in the current step, but by everything it has seen in the past as well. This makes them far more flexible than a vanilla neural network. In practice, most people use a Long Short-Term Memory (LSTM) model, which is a refied version of a RNN that handles these long range dependencies better. One application of these types of models is a character-level language model that attempts to predict the next character in a sequence. Through the training process, the model can learn to generate new text one character at a time, learning spellilng, puncuation, and structure from raw data with now explicit rules provided. 

The article then discusses 5 example character models trained on 5 very different datasets in order to demonstrate the power of a character-level RNN. Karpathy begins with a model trained on Paul Graham's essays, which produced plausible-sounding passages with self citations. However, it is difficult to get any real meaning from the generated text. He then trains a model using Shakespear, which learns to generate dialogue with character names, stage directions, and something close to iambic pentameter. Training on Wikipedia then allows the model to train on a structured markdown language instead of just on english. The wikipedia model was able to generate random but valid XML. To step it up, they then train on raw Latex source files to see how well the model can learn complicated markdown languages. It struggles a little bit to match the true structure of latex, for example it will open commands such as \begin{proof} but then end it with \end{lemma}, however it still learned a general plausible structure. Karpathy then tries to push structured data to its limit by training a model on code. At a glance, the resulting code looks real, although it of course doesn't compile when run. Finally, Karpathy tries to train a model to generate baby names from a text file containing 8000 baby names. Again, the model outputs text that seems plausible at first, but with closer inspection is a bit off. With more training data and computational power, these models are likely capable of much more. 

Beyond the impressive outputs from the models, Karpathy also offers insight into why these models work as well as they do. By visualizing the activations of individual neuron in a trained RNN, he discovers that some cells learn surprisingly specific and interpretable behaviors. For example, one of the markdown models he trained had a neuron that would activate inside URLs, another that tracks whether the model is inside a double bracket markdown environment, and another that appeared to count characters in a repeating pattern like "www". These behaviors were not directly programmed, they were learned by the model because tracking these patterns turned out to be useful for predicting the next character. Looking ahead, Karpathy points to several different directions in RNN research. Primarily, there is extensive research in the concept of attention, which allows a model to selevtively focus on relevant parts of its input rather than compressing everything into a single hidden state. He also points out that RNNs are quickly becoming used in computer vision for frame-level video classification, image captioning, video captioning, and visual question answering. Combined with external memory mechanisms explored in Neural Turing Machines, these ideas pointed toward a future of more flexible, powerful, and generalizable sequence models

## Problem-3: 

Write code for a character-based RNN (i.e. LSTM or GRU, not simple_RNN) in PyTorch. You can use his source code and any code online to assist. If you do, just reference where you sought help from.

- Choose a large text corpus, such as a collection of novels from [Project Gutenburg](https://www.gutenberg.org/). You can use other data, but you should explain where your data comes from.

> I chose to download every novel by Fyodor Dostoyevsky

- Perform any necessary preprocessing, explaining what steps you take. In particular, what forms of normalization do you use? Do you define characters with special meaning?

In [1]:
# Load packages
import math
import pandas as pd 
import nltk 
import re
from pathlib import Path
import itertools

import torch
import torch.nn as nn 
from torch.utils.data import DataLoader, TensorDataset, random_split

In [2]:
# Function to remove header
def clean_gutenberg(text):
    start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
    end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"
    
    start_idx = text.find(start_marker)
    end_idx = text.find(end_marker)
    
    if start_idx != -1:
        # Move past the marker line itself
        start_idx = text.find("\n", start_idx) + 1
    else:
        start_idx = 0  
    
    if end_idx != -1:
        text = text[start_idx:end_idx]
    else:
        text = text[start_idx:] 
    
    return text.strip()

# Import books 
text = ""
for filepath in Path('dostoyevsky/').glob('*.txt'):
    with open(filepath, 'r', encoding='utf-8') as f:
        raw = f.read()
    
    cleaned = clean_gutenberg(raw)
    text += cleaned + "\n"

print(f"\nTotal characters: {len(text):,}")

### Pre-Processing 
text = text.lower()
# text = re.sub(r'[^\x00-\x7F]', '', text)

## TEMPORARY subsample
# text = text[:500_000]

## Build character vocabulary
chars = sorted(set(text)) 
data_size, vocab_size = len(text), len(chars)
print(f'Data has {data_size} characters, {vocab_size} unique characters.')

## Mappings
char_to_ix = {ch:i for i, ch in enumerate(chars)}
ix_to_char = {i:ch for i, ch in enumerate(chars)}

## Encode text as integers
encoded = [char_to_ix[ch] for ch in text]


Total characters: 4,418,163
Data has 4418163 characters, 71 unique characters.


> The only normalization steps are converting the text to lower case
> 
> No special characters were defined. The vocabulary consists entirely of characters that appear naturally in the text after normalization including lowercase ASCII letters, digits, punctuation, and whitespace. Newline characters (\n) are left in the vocabulary and carry implicit meaning as paragraph/line boundaries.

- Efficiently load and batch the dataset for training using a `DataLoader`. Make sure to reserve some of the data for validation and testing. Describe how you handle batching and sequence lengths.

In [3]:
## Create input/target sequences
seq_length = 100  
stride = seq_length

n          = len(encoded)
train_end  = int(0.80 * n)
val_end    = int(0.90 * n)

splits = {
    "train": encoded[:train_end],
    "val":   encoded[train_end:val_end],
    "test":  encoded[val_end:],
}

def make_dataset(enc, seq_length, stride):
    """Return a TensorDataset of (input_seq, target_seq) pairs."""
    inputs, targets = [], []
    for i in range(0, len(enc) - seq_length, stride):
        inputs.append(enc[i : i + seq_length])
        targets.append(enc[i + 1 : i + seq_length + 1])
    X = torch.tensor(inputs, dtype=torch.long)
    y = torch.tensor(targets, dtype=torch.long)
    return TensorDataset(X, y)

train_dataset = make_dataset(splits["train"], seq_length, stride)
val_dataset   = make_dataset(splits["val"],   seq_length, stride)
test_dataset  = make_dataset(splits["test"],  seq_length, stride)

print(f"Train sequences : {len(train_dataset):,}")
print(f"Val   sequences : {len(val_dataset):,}")
print(f"Test  sequences : {len(test_dataset):,}")
 
batch_size   = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

Train sequences : 35,345
Val   sequences : 4,418
Test  sequences : 4,418


> The batch size is set to 64, meaning the model sees 64 sequences at once per training step rather than one at a time. The shuffle=True option means batches are randomly assembled each epoch, which helps the model generalize. 
>
> The sequence length is set to 100, meaning every input sequence is exactly 100 characters long. This ensures all inputs are the same length, eliminating the need for padding. 


- Define your RNN model. Discuss the number of layers, hidden units, and the type of RNN cells you use. What is the total number of parameters in your model? Explain the rationale behind your architectural choices.

In [4]:
# Define LSTM class
class LSTMModel(nn.Module):
    def __init__(self, vocab_dim, embed_dim, hidden_dim, layer_dim, dropout=0.3):
        super(LSTMModel, self).__init__()
        self.device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.hidden_dim = hidden_dim
        self.layer_dim  = layer_dim

        self.embedding = nn.Embedding(vocab_dim,embed_dim)
        self.lstm      = nn.LSTM(
            embed_dim, hidden_dim, layer_dim, 
            batch_first=True, 
            dropout=dropout if layer_dim > 1 else 0.0
        )
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_dim, vocab_dim)

    def forward(self, x, hidden=None):
        if hidden is None:
            h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim, device=self.device)
            c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim, device=self.device)
            hidden = (h0, c0)
 
        out, hidden = self.lstm(self.embedding(x), hidden)
        logits      = self.fc(self.drop(out))         
        return logits, hidden

In [5]:
# CPU/GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:",device)

# Intialize model, loss function, and optimizer
model = LSTMModel(vocab_dim  = vocab_size, 
                  embed_dim  = 32, 
                  hidden_dim = 512, 
                  layer_dim  = 2,
                  dropout    = 0.3
                  ).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=2, factor=0.5
)

Device: cuda


> I chose a LSTM model for this exercise since they handle long-range dependencies well and are typically used over a vanilla RNN. I originally used a learning rate of 0.01, however this caused the training to jump out of a local minima so I reduced it, leading to better results. The other parameters (embedding dimension, hidden dimension, and layer dimension) were initially chosen somewhat arbitrarily, but then were finalized after the hyper-parameter search later in this notebook. 
> 
> Source code: https://www.geeksforgeeks.org/deep-learning/long-short-term-memory-networks-using-pytorch/ 


- Write the training loop.

In [6]:
# Train
num_epochs = 30
best_val = float("inf")

for epoch in range(1, num_epochs+1):
    # Training 
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        logits, _ = model(X_batch) 

        loss = criterion(logits.view(-1, vocab_size), y_batch.view(-1))
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  
        optimizer.step()
        
        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            logits, _ = model(X_batch)
            val_loss += criterion(logits.view(-1, vocab_size), y_batch.view(-1)).item()

    # Logging
    avg_train = train_loss / len(train_loader)
    avg_val   = val_loss / len(val_loader)
    print(f"Epoch {epoch}/{num_epochs} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")

    scheduler.step(avg_val)
    
    if avg_val < best_val:
        best_val = avg_val
        torch.save(model.state_dict(), "best_lstm.pt")
 
print(f"\nBest val loss: {best_val:.4f}  (saved → best_lstm.pt)")


Epoch 1/30 | Train Loss: 2.1456 | Val Loss: 1.6805
Epoch 2/30 | Train Loss: 1.5921 | Val Loss: 1.4492
Epoch 3/30 | Train Loss: 1.4336 | Val Loss: 1.3518
Epoch 4/30 | Train Loss: 1.3529 | Val Loss: 1.3012
Epoch 5/30 | Train Loss: 1.3041 | Val Loss: 1.2694
Epoch 6/30 | Train Loss: 1.2696 | Val Loss: 1.2474
Epoch 7/30 | Train Loss: 1.2437 | Val Loss: 1.2292
Epoch 8/30 | Train Loss: 1.2233 | Val Loss: 1.2186
Epoch 9/30 | Train Loss: 1.2063 | Val Loss: 1.2086
Epoch 10/30 | Train Loss: 1.1925 | Val Loss: 1.2049
Epoch 11/30 | Train Loss: 1.1798 | Val Loss: 1.1989
Epoch 12/30 | Train Loss: 1.1691 | Val Loss: 1.1937
Epoch 13/30 | Train Loss: 1.1560 | Val Loss: 1.1813
Epoch 14/30 | Train Loss: 1.1441 | Val Loss: 1.1767
Epoch 15/30 | Train Loss: 1.1356 | Val Loss: 1.1753
Epoch 16/30 | Train Loss: 1.1270 | Val Loss: 1.1714
Epoch 17/30 | Train Loss: 1.1194 | Val Loss: 1.1697
Epoch 18/30 | Train Loss: 1.1123 | Val Loss: 1.1687
Epoch 19/30 | Train Loss: 1.1065 | Val Loss: 1.1687
Epoch 20/30 | Train L


- Monitor and report on the training progress by tracking the loss. After training, evaluate the model's performance using a suitable evaluation metric (e.g., perplexity) on a validation dataset or a held-out portion of the training data. Discuss the results.

In [7]:
def evaluate_perplexity(model, loader, criterion, device, vocab_size):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits, _ = model(X_batch)
            
            # Sum loss so we can average over tokens correctly
            loss = criterion(logits.view(-1, vocab_size), y_batch.view(-1))
            total_loss   += loss.item() * y_batch.numel()
            total_tokens += y_batch.numel()

    avg_loss    = total_loss / total_tokens
    perplexity  = math.exp(avg_loss)
    return avg_loss, perplexity

# Load best checkpoint
model.load_state_dict(torch.load("best_lstm.pt", map_location=device))

# Evaluate on all three splits
for split_name, loader in [("Train", train_loader), 
                            ("Val",   val_loader), 
                            ("Test",  test_loader)]:
    loss, ppl = evaluate_perplexity(model, loader, criterion, device, vocab_size)
    print(f"{split_name:>5} | Loss: {loss:.4f} | Perplexity: {ppl:.2f}")

print(f"\nRandom baseline perplexity: {vocab_size}")

Train | Loss: 0.9918 | Perplexity: 2.70
  Val | Loss: 1.1690 | Perplexity: 3.22
 Test | Loss: 1.1619 | Perplexity: 3.20

Random baseline perplexity: 71


> A perplexity of around 3.2 means the model on average narrows down a prediction to 3.2 characters. This is down from 71 from the baseline before the model was trained. 

- Specify the hyper-parameters (for example, model hyper-parameters, as well as sampling size and beam width from below) used in your model. Find suitable settings using a validation split.

In [8]:
# Hyperparameter grid
param_grid = {
    "hidden_dim": [128, 256, 512],
    "layer_dim": [1,2],
    "embed_dim": [32, 64]
}

num_search_epochs = 5
batch_size = 64

def train_and_eval(config, train_loader, val_loader, vocab_size, device):
    model = LSTMModel(
        vocab_dim  = vocab_size, 
        embed_dim  = config["embed_dim"], 
        hidden_dim = config["hidden_dim"], 
        layer_dim  = config["layer_dim"],
        dropout    = 0.3
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=1, factor=0.5
    )

    best_val_ppl = float("inf")

    for epoch in range(1, num_search_epochs+1):
        # Train 
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            logits, _ = model(X_batch)

            loss = criterion(logits.view(-1, vocab_size), y_batch.view(-1))
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        
        # Validate 
        model.eval()
        total_loss, total_tokens = 0.0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)

                logits, _ = model(X_batch)
                loss = criterion(logits.view(-1, vocab_size), y_batch.view(-1))

                total_loss   += loss.item() * y_batch.numel()
                total_tokens += y_batch.numel()
        
        val_ppl = math.exp(total_loss / total_tokens)
        scheduler.step(total_loss / total_tokens)
        best_val_ppl = min(best_val_ppl, val_ppl)

    return best_val_ppl, model


# --- Run grid search ---
keys   = list(param_grid.keys())
combos = list(itertools.product(*param_grid.values()))

results = []
for i, values in enumerate(combos):
    config = dict(zip(keys, values))
    print(f"[{i+1}/{len(combos)}] {config}")
    ppl, _ = train_and_eval(config, train_loader, val_loader, vocab_size, device)
    results.append((ppl, config))
    print(f"         → Val PPL: {ppl:.2f}\n")

results.sort(key=lambda x: x[0])

best_config = results[0][1]
print(f"\nBest config: {best_config}")

[1/12] {'hidden_dim': 128, 'layer_dim': 1, 'embed_dim': 32}
         → Val PPL: 5.26

[2/12] {'hidden_dim': 128, 'layer_dim': 1, 'embed_dim': 64}
         → Val PPL: 5.09

[3/12] {'hidden_dim': 128, 'layer_dim': 2, 'embed_dim': 32}
         → Val PPL: 4.92

[4/12] {'hidden_dim': 128, 'layer_dim': 2, 'embed_dim': 64}
         → Val PPL: 4.88

[5/12] {'hidden_dim': 256, 'layer_dim': 1, 'embed_dim': 32}
         → Val PPL: 4.43

[6/12] {'hidden_dim': 256, 'layer_dim': 1, 'embed_dim': 64}
         → Val PPL: 4.28

[7/12] {'hidden_dim': 256, 'layer_dim': 2, 'embed_dim': 32}
         → Val PPL: 4.19

[8/12] {'hidden_dim': 256, 'layer_dim': 2, 'embed_dim': 64}
         → Val PPL: 4.01

[9/12] {'hidden_dim': 512, 'layer_dim': 1, 'embed_dim': 32}
         → Val PPL: 3.74

[10/12] {'hidden_dim': 512, 'layer_dim': 1, 'embed_dim': 64}
         → Val PPL: 3.73

[11/12] {'hidden_dim': 512, 'layer_dim': 2, 'embed_dim': 32}
         → Val PPL: 3.41

[12/12] {'hidden_dim': 512, 'layer_dim': 2, 'embed_d

In [ ]:
# Tune sampling temperature
# Evaluate temperature via val-set perplexity under each sampling distribution
def temperature_val_ppl(model, val_loader, vocab_size, device, temperature):
    """
    Rescale logits by temperature before computing loss -
    this reflects the distribution actually used during sampling.
    """
    model.eval()
    criterion  = nn.CrossEntropyLoss()
    total_loss, total_tokens = 0.0, 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits, _ = model(X_batch)
            scaled_logits = logits / temperature
            loss = criterion(scaled_logits.view(-1, vocab_size), y_batch.view(-1))
            total_loss   += loss.item() * y_batch.numel()
            total_tokens += y_batch.numel()

    return math.exp(total_loss / total_tokens)


temperatures = [0.4, 0.6, 0.8, 1.0, 1.2, 1.5]
print("*** Temperature search ***")
temp_results = []
for t in temperatures:
    ppl = temperature_val_ppl(model, val_loader, vocab_size, device, t)
    temp_results.append((t, ppl))
    print(f"  Temperature {t:.1f} → Val PPL: {ppl:.2f}")

best_temp = min(temp_results, key=lambda x: x[1])[0]
print(f"\nBest temperature: {best_temp}")

*** Temperature search ***
  Temperature 0.4 → Val PPL: 5.68
  Temperature 0.6 → Val PPL: 3.88
  Temperature 0.8 → Val PPL: 3.38
  Temperature 1.0 → Val PPL: 3.22
  Temperature 1.2 → Val PPL: 3.21
  Temperature 1.5 → Val PPL: 3.33

Best temperature: 1.2


> From these results, it appears that a temperature of 1.0 (the natural distribution) is the best choice for this model.

In [ ]:
def beam_search_generate(model, seed_text, length, beam_width,
                          char_to_ix, ix_to_char, device):
    model.eval()
    chars_in = [char_to_ix.get(ch, 0) for ch in seed_text.lower()]
    inp      = torch.tensor([chars_in], dtype=torch.long, device=device)

    with torch.no_grad():
        _, hidden = model(inp)
        x = inp[:, -1:]

        # Each beam: (log_prob, sequence_string, hidden_state)
        init_logits, init_hidden = model(x, hidden)
        log_probs = torch.log_softmax(init_logits[0, -1], dim=0)
        topk      = torch.topk(log_probs, beam_width)

        beams = [
            (topk.values[i].item(),
             seed_text + ix_to_char[topk.indices[i].item()],
             init_hidden,
             topk.indices[i].item())
            for i in range(beam_width)
        ]

        for _ in range(length - 1):
            candidates = []
            for score, seq, hid, last_idx in beams:
                x_in = torch.tensor([[last_idx]], dtype=torch.long, device=device)

                # Unpack and clone hidden state per beam
                h, c   = hid
                logits, new_hidden = model(x_in, (h.clone(), c.clone()))
                lp     = torch.log_softmax(logits[0, -1], dim=0)
                topk_b = torch.topk(lp, beam_width)

                for i in range(beam_width):
                    candidates.append((
                        score + topk_b.values[i].item(),
                        seq + ix_to_char[topk_b.indices[i].item()],
                        new_hidden,
                        topk_b.indices[i].item()
                    ))

            # Keep top beam_width candidates
            candidates.sort(key=lambda x: x[0], reverse=True)
            beams = candidates[:beam_width]

    return beams[0][1]   # return highest-scoring sequence


def distinct_ngrams(text, n):
    """Fraction of unique n-grams - proxy for output diversity."""
    ngrams = [text[i:i+n] for i in range(len(text) - n)]
    return len(set(ngrams)) / len(ngrams) if ngrams else 0.0


SEED   = "the old man looked"
LENGTH = 300
beam_widths = [1, 2, 4, 8, 16]

print(" Beam width search")
print(f"{'Width':>6} | {'Distinct-2':>10} | {'Distinct-3':>10}")
print("-" * 35)

beam_results = []
for bw in beam_widths:
    text_out = beam_search_generate(
        model, SEED, LENGTH, bw, char_to_ix, ix_to_char, device
    )
    d2 = distinct_ngrams(text_out, 2)
    d3 = distinct_ngrams(text_out, 3)
    beam_results.append((bw, d2, d3, text_out))
    print(f"{bw:>6} | {d2:>10.3f} | {d3:>10.3f}")

# Print generated text per beam width for qualitative inspection
print("\n Beam search samples")
for bw, d2, d3, text_out in beam_results:
    print(f"\n  [Beam width = {bw}]")
    print("  " + text_out)

 Beam width search
 Width | Distinct-2 | Distinct-3
-----------------------------------
     1 |      0.288 |      0.432
     2 |      0.383 |      0.568
     4 |      0.310 |      0.457
     8 |      0.297 |      0.441
    16 |      0.380 |      0.527

 Beam search samples

  [Beam width = 1]
  the old man looked and sat down again.

“i shall be a strange and more than that i am a strange and more than that i am
always allowed to see you again. i was a strange and more than that i
am a stranger and the same and soul and the same and the same and
solitude and the same and solitude. i was a strange and more
s

  [Beam width = 2]
  the old man lookeded at the prince’s hand.

“what do you mean?” said the prince, as though he were all that had been
absolutely anxious to say that the prince was standing in the corner of
the room, and the prince was standing in the street. he was standing
at the table and stood at the table and stood at the table.


  [Beam width = 4]
  the old man looked in

> Based on these results, a beam width = 2 is the best choice. Everything above 2 fell into repetitive loops. 

- Implement a text generation function using the trained RNN model. Provide a prompt or seed text, and use the RNN to generate a sequence of characters. Experiment with different prompt texts and observe how the generated text changes. Discuss any interesting patterns or observations you make during the text generation process.

In [ ]:
# Generation Functions 
def generate_sample(model, seed_text, length, temperature, char_to_ix, ix_to_char, device):
    model.eval()
    chars_in = [char_to_ix.get(ch, 0) for ch in seed_text.lower()]
    inp      = torch.tensor([chars_in], dtype=torch.long, device=device)

    with torch.no_grad():
        _, hidden = model(inp)
        x = inp[:, -1:]
        generated = seed_text

        for _ in range(length):
            logits, hidden = model(x, hidden)
            probs = torch.softmax(logits[0, -1] / temperature, dim=0)
            idx   = torch.multinomial(probs, 1).item()
            generated += ix_to_char[idx]
            x = torch.tensor([[idx]], dtype=torch.long, device=device)

    return generated

# Load best model 
model.load_state_dict(torch.load("best_lstm.pt", map_location=device))
model.eval()

### Experiment 1: Different Prompt Styles 
# Test how well the model continues different types of opening

prompts = {
    "Narrator (scene setting)" : "the room was dark and silent, and",
    "Dialogue opening"         : '"you must understand," he said,',
    "Character intro"          : "raskolnikov stood at the corner of",
    "Emotional state"          : "she felt a sudden wave of",
    "Out-of-distribution"      : "the spaceship landed quietly on",   # nothing like this in training data
}

TEMP       = 1.2    # best temperature from earlier search
BW         = 2      # best beam width from earlier search
LENGTH     = 300

print("=" * 65)
print("EXPERIMENT 1: Different Prompt Styles")
print("=" * 65)
for label, prompt in prompts.items():
    out = generate_sample(model, prompt, LENGTH, TEMP, char_to_ix, ix_to_char, device)
    print(f"\n── {label}")
    print(f"   Prompt : '{prompt}'")
    print(f"   Output : {out}")

### Experiment 2: Prompt Length 
# Short prompts give the model little context to prime its hidden
# state; longer prompts give it more to work with

base_prompt_short  = "he"
base_prompt_medium = "he walked slowly through"
base_prompt_long   = "he walked slowly through the narrow streets, thinking about what had happened the night before, and wondering whether"

print("\n" + "=" * 65)
print("EXPERIMENT 2: Prompt Length Effect")
print("=" * 65)
for prompt in [base_prompt_short, base_prompt_medium, base_prompt_long]:
    out = generate_sample(model, prompt, LENGTH, TEMP, char_to_ix, ix_to_char, device)
    print(f"\n── Prompt length: {len(prompt)} chars")
    print(f"   Prompt : '{prompt}'")
    print(f"   Output : {out}")

### Experiment 3: Sampling vs Beam Search 
# Direct comparison on the same prompt

prompt = "the old man looked at him and"

EXPERIMENT 1: Different Prompt Styles (temperature sampling)

── Narrator (scene setting)
   Prompt : 'the room was dark and silent, and'
   Output : the room was dark and silent, and quietly
took on fainting and easily, though seeming jumped up
and deny, he had no more accurtant in the mirfer of a serious age disbitches from
the article; on her gratitude was apparently drongers, to
become at every action being penitent. lot of paris, bitter things,
dmitri fyodorovitch, lie from

── Dialogue opening
   Prompt : '"you must understand," he said,'
   Output : "you must understand," he said, that’s the elder. yes, my blood! why from children?”

marmeladov started. “now what if you did anything every day.” my little
report, plunged in a fool, with meritch.... now
langed some “original, that is, then otherwise that), there are
young lady, when he even always believed that riffuan, he dre

── Character intro
   Prompt : 'raskolnikov stood at the corner of'
   Output : raskolnikov stood at the

> The output from the first experiment works reasonably well, but still is fairly non-sensical. The dialogue opening produces the most coherent output, which makes sense since Dostoyevsky's novels are heavily dialogue-driven. I also notice that a lot of the other prompts end up having dialogue in the outputted text, as expected. 
>
> Interestingly, when varying prompt length, longer prompts don't obviously produce better output. The 2-character prompt actually generates a slightly more coherent passage than the others. This suggests the model's hidden state gets primed adequately even from short prompts, and that beyond a certain length, additional context doesn't help much.

## Problem-4: Optional extra credit 
 
Up to +3 bonus points 

* Repeat the text generation process from the previous problem, but do it with a transformer architecture rather than an LSTM/GRU (you can use word tokens instead of character tokens if you prefer). Use the provided mini-GPT lab for reference.  

## Problem-5: Optional extra credit 
 


- (+1 bonus points): Now with your model trained, implement top-k and nucleus sampling. Again, provide a prompt or seed text, and use the RNN to generate a sequence of characters. Experiment with different prompt texts and observe how the generated text changes. Discuss any interesting patterns or observations you make during the text generation process.

In [12]:
# INSERT CODE

- (+1 bonus points): Now with your model trained, implement beam search. Again, provide a prompt or seed text, and use the RNN to generate a sequence of characters. Experiment with different prompt texts and observe how the generated text changes. Discuss any interesting patterns or observations you make during the text generation process.  

In [13]:
# INSERT CODE